# Quickstart + Interpretation

### 1. Imports

In [1]:
from HMM import gaussianHMM, plot_regimes
import yfinance as yf
import numpy as np

### 2. Data Fetching

In [2]:
spy = yf.download('^GSPC', start='2000-12-31', end='2026-12-31')
spy_train = spy.loc['2000-12-31':'2020-12-31']
spy_test = spy.loc['2020-12-31':'2026-12-31'] # to today

[*********************100%***********************]  1 of 1 completed


In [3]:
# roughly a 80/20 train-test split
returns_train = np.log(spy_train['Close'] / spy_train['Close'].shift(1)).dropna()
close_train = spy_train['Close'].dropna()

returns_test = np.log(spy_test['Close'] / spy_test['Close'].shift(1)).dropna()
close_test = spy_test['Close'].dropna()

### 3. Model Fitting

In [4]:
hmm = gaussianHMM(n_states=3)

In [5]:
hmm.fit(returns_train, sort = 'sharpe')

Initial Parameters:
pi: [0.3333 0.3333 0.3333]
A: [[0.6 0.2 0.2]
 [0.2 0.6 0.2]
 [0.2 0.2 0.6]]
Initial State Specific Parameters:
State 0: mu = -0.0113, sigma = 0.0108
State 1: mu = 0.0007, sigma = 0.0017
State 2: mu = 0.0113, sigma = 0.0095
Commencing iterative estimation...
Iteration 1: Log Likelihood: 14623.6324
Iteration 2: Log Likelihood: 15256.8060
Iteration 3: Log Likelihood: 15436.2559
Iteration 4: Log Likelihood: 15602.9441
Iteration 5: Log Likelihood: 15760.1429
Iteration 6: Log Likelihood: 15859.0256
Iteration 7: Log Likelihood: 15925.0636
Iteration 8: Log Likelihood: 15973.0964
Iteration 9: Log Likelihood: 16015.5891
Iteration 10: Log Likelihood: 16055.2619
Iteration 11: Log Likelihood: 16092.7162
Iteration 12: Log Likelihood: 16127.8325
Iteration 13: Log Likelihood: 16160.2119
Iteration 14: Log Likelihood: 16189.1883
Iteration 15: Log Likelihood: 16213.7407
Iteration 16: Log Likelihood: 16232.8345
Iteration 17: Log Likelihood: 16246.4639
Iteration 18: Log Likelihood: 1625

In [6]:
viterbi_states = hmm.predict(returns_train, type = "viterbi")
probability_states = hmm.predict(returns_train, type = "probability")
print(viterbi_states)

[0 0 0 ... 2 2 2]


### 4. Model Validation

In [8]:
plot_regimes(price = close_train, 
             regimes = viterbi_states, 
             hmm=hmm, 
             gamma = probability_states, 
             returns = returns_train,
             title="S&P 500 — HMM Regimes Training from 31-12-2000 to 31-12-2020")

Seems to identify regimes decently well. Brown regime seems to be slightly downwards drifting, potential left-skewness at "neutral" regime.

Dotcom bubble 2002 , the GFC 2008, European Debt Crisis 2011, Covid 19 2020 has been captured.

However, the Great Fall of China in 2016 was not.

### 5. Model Test

In [9]:
viterbi_states_test = hmm.predict(returns_test, type = "viterbi")
probability_states_test = hmm.predict(returns_test, type = "probability")
posterior_states_test = hmm.predict(returns_test, type = "posterior")

In [10]:
plot_regimes(price = close_test, 
             regimes = viterbi_states_test, 
             hmm = hmm, 
             gamma = probability_states_test, 
             returns = returns_test,
             title="S&P 500 — HMM Regimes Test from 31-12-2020 to Today")

The Russia-Ukraine War 2022, Liberation Day 2025 was captured.

Iran War today was not captured.